# S4_03 — The Full RAG Flow with `VectorIndex`

> **Skilljar source**: Lesson **L05 — Implementing the RAG flow** (L04 "The full RAG flow" is theory-only, covered conceptually in Week_05.md §1.4)
> **Week_05.md mapping**: §1.4 "The Full RAG Flow" + §1.5 "Implementing the RAG Flow"
> **Original file**: `_originals_skilljar/003_vectordb.ipynb`

## What You Will Learn

1. How to encapsulate embedding + storage + search into a single reusable `VectorIndex` class.
2. How cosine distance powers semantic retrieval — the core of RAG.
3. The **6-step RAG pipeline** (Week_05.md §1.4):
   1. Chunk source text
   2. Embed each chunk
   3. Store vectors in an index
   4. Embed the user query
   5. Find top-k nearest chunks
   6. Inject chunks into Claude's prompt

## ✅ This Notebook Is Complete

The code cells below are filled in from Week_05.md. If you prefer to solve the exercises yourself, open the companion practice version at `_student_practice_20260420/S4_03_vector_search.ipynb` — it has the same structure but with the Step 2 / 3 / 5 / 6 code cells left blank for you to fill in.

> [!tip] Bug-fix note
> The Skilljar original used `store.add_documents([...])` in Step 4, but the `VectorIndex` class does not define that method — only `add_document` (singular) and `add_vector`. This notebook follows Week_05.md §1.5 L727-729 and uses an explicit `for`-loop with `add_vector` instead. The result is identical and the code matches the class exactly.

## Connection to Week_05.md

| Week_05.md Section | This Notebook |
|---|---|
| §1.5 "VectorIndex class" + cosine formula | Cell 8 — class definition |
| §1.5 L709 "Step 1: Chunk source text" | Cell 12 — `chunk_by_section(text)` |
| §1.5 L718 "Step 2: Embed chunks" | Cell 14 — `generate_embedding(chunks)` |
| §1.5 L727-729 "Step 3: Store vectors" | Cell 16 — `for (emb, chunk): store.add_vector(...)` |
| §1.5 L740 "Step 4: Embed user query" | Cell 18 — `user_embedding = generate_embedding(...)` |
| §1.5 L748-751 "Step 5: Similarity search" | Cell 20 — `results = store.search(user_embedding, 2)` |

## Setup · VoyageAI Client

Identical to S4_02. Loading `.env` and creating the client.

In [ ]:
# Client Setup
from dotenv import load_dotenv
import voyageai

load_dotenv()

client = voyageai.Client()

## Reuse · `chunk_by_section` from S4_01

Re-declared here for self-containment.

In [ ]:
# Chunk by section
import re


def chunk_by_section(document_text):
    pattern = r"\n## "
    return re.split(pattern, document_text)

## Reuse · `generate_embedding` from S4_02

**Note the subtle upgrade vs S4_02**: this version supports **both** a single string AND a list of strings (batch embedding). Batching is critical later to avoid VoyageAI rate-limiting when indexing many chunks at once.

> [!tip] Week_05.md §1.3
> In production you almost always want batched embedding — one HTTP round-trip per chunk is wasteful.

In [ ]:
# Embedding Generation
def generate_embedding(chunks, model="voyage-3-large", input_type="query"):
    is_list = isinstance(chunks, list)
    input = chunks if is_list else [chunks]
    result = client.embed(input, model=model, input_type=input_type)
    return result.embeddings if is_list else result.embeddings[0]

## Core Component · `VectorIndex` Class

This is the centerpiece of S4_03. It combines three responsibilities in one object:

1. **Storage** — holds `(vector, document)` pairs
2. **Addition** — `add_document(doc)` embeds + stores; `add_vector(v, doc)` stores a precomputed vector
3. **Search** — `search(query, k)` returns the top-k closest documents by cosine distance

### Interface
```python
store = VectorIndex(embedding_fn=generate_embedding)
store.add_document({"content": "some text"})
store.search("user query", k=5)
```

> [!finding] Week_05.md §1.5 — Design choices
> - `distance_metric` is a parameter (`"cosine"` default) — swap to `"euclidean"` without changing user code.
> - `embedding_fn` is dependency-injected — you can mock it in tests, or plug a different provider.
> - `add_vector()` lets advanced users precompute vectors (e.g., with a GPU) and inject them.

### Cosine distance refresher (Week_05.md §1.5)
```
cosine_similarity = (a · b) / (||a|| · ||b||)      ∈ [-1, +1]
cosine_distance   = 1 - cosine_similarity          ∈ [0, 2]
```
Smaller distance ⇒ more similar. The `search` method sorts **ascending by distance**.

In [ ]:
# VectorIndex implementation
import math
from typing import Optional, Any, List, Dict, Tuple


class VectorIndex:
    def __init__(
        self,
        distance_metric: str = "cosine",
        embedding_fn=None,
    ):
        self.vectors: List[List[float]] = []
        self.documents: List[Dict[str, Any]] = []
        self._vector_dim: Optional[int] = None
        if distance_metric not in ["cosine", "euclidean"]:
            raise ValueError("distance_metric must be 'cosine' or 'euclidean'")
        self._distance_metric = distance_metric
        self._embedding_fn = embedding_fn

    def add_document(self, document: Dict[str, Any]):
        if not self._embedding_fn:
            raise ValueError(
                "Embedding function not provided during initialization."
            )
        if not isinstance(document, dict):
            raise TypeError("Document must be a dictionary.")
        if "content" not in document:
            raise ValueError(
                "Document dictionary must contain a 'content' key."
            )

        content = document["content"]
        if not isinstance(content, str):
            raise TypeError("Document 'content' must be a string.")

        vector = self._embedding_fn(content)
        self.add_vector(vector=vector, document=document)

    def search(
        self, query: Any, k: int = 1
    ) -> List[Tuple[Dict[str, Any], float]]:
        if not self.vectors:
            return []

        if isinstance(query, str):
            if not self._embedding_fn:
                raise ValueError(
                    "Embedding function not provided for string query."
                )
            query_vector = self._embedding_fn(query)
        elif isinstance(query, list) and all(
            isinstance(x, (int, float)) for x in query
        ):
            query_vector = query
        else:
            raise TypeError(
                "Query must be either a string or a list of numbers."
            )

        if self._vector_dim is None:
            return []

        if len(query_vector) != self._vector_dim:
            raise ValueError(
                f"Query vector dimension mismatch. Expected {self._vector_dim}, got {len(query_vector)}"
            )

        if k <= 0:
            raise ValueError("k must be a positive integer.")

        if self._distance_metric == "cosine":
            dist_func = self._cosine_distance
        else:
            dist_func = self._euclidean_distance

        distances = []
        for i, stored_vector in enumerate(self.vectors):
            distance = dist_func(query_vector, stored_vector)
            distances.append((distance, self.documents[i]))

        distances.sort(key=lambda item: item[0])

        return [(doc, dist) for dist, doc in distances[:k]]

    def add_vector(self, vector, document: Dict[str, Any]):
        if not isinstance(vector, list) or not all(
            isinstance(x, (int, float)) for x in vector
        ):
            raise TypeError("Vector must be a list of numbers.")
        if not isinstance(document, dict):
            raise TypeError("Document must be a dictionary.")
        if "content" not in document:
            raise ValueError(
                "Document dictionary must contain a 'content' key."
            )

        if not self.vectors:
            self._vector_dim = len(vector)
        elif len(vector) != self._vector_dim:
            raise ValueError(
                f"Inconsistent vector dimension. Expected {self._vector_dim}, got {len(vector)}"
            )

        self.vectors.append(list(vector))
        self.documents.append(document)

    def _euclidean_distance(
        self, vec1: List[float], vec2: List[float]
    ) -> float:
        if len(vec1) != len(vec2):
            raise ValueError("Vectors must have the same dimension")
        return math.sqrt(sum((p - q) ** 2 for p, q in zip(vec1, vec2)))

    def _dot_product(self, vec1: List[float], vec2: List[float]) -> float:
        if len(vec1) != len(vec2):
            raise ValueError("Vectors must have the same dimension")
        return sum(p * q for p, q in zip(vec1, vec2))

    def _magnitude(self, vec: List[float]) -> float:
        return math.sqrt(sum(x * x for x in vec))

    def _cosine_distance(self, vec1: List[float], vec2: List[float]) -> float:
        if len(vec1) != len(vec2):
            raise ValueError("Vectors must have the same dimension")

        mag1 = self._magnitude(vec1)
        mag2 = self._magnitude(vec2)

        if mag1 == 0 and mag2 == 0:
            return 0.0
        elif mag1 == 0 or mag2 == 0:
            return 1.0

        dot_prod = self._dot_product(vec1, vec2)
        cosine_similarity = dot_prod / (mag1 * mag2)
        cosine_similarity = max(-1.0, min(1.0, cosine_similarity))

        return 1.0 - cosine_similarity

    def __len__(self) -> int:
        return len(self.vectors)

    def __repr__(self) -> str:
        has_embed_fn = "Yes" if self._embedding_fn else "No"
        return f"VectorIndex(count={len(self)}, dim={self._vector_dim}, metric='{self._distance_metric}', has_embedding_fn='{has_embed_fn}')"

## Step 1 · Load the Source Document

Plain file read — `report.md` is our gold-standard corporate annual report shared across all S4 notebooks.

In [ ]:
with open("./report.md", "r") as f:
    text = f.read()

## Step 2 · Chunk the Text by Section

Split the raw report into one chunk per Markdown `## ` section.

**Week_05.md §1.5 reference**: line 709 (`chunks = chunk_by_section(text)`).

> [!tip] Why `chunks[2]`?
> The third chunk typically contains the document's table of contents or the first real section. Printing it is a quick sanity check — Week_05.md line 710 does the same (`chunks[2]  # Test to see the table of contents`).

In [ ]:
# Step 1: Chunk the text by section  (Week_05.md §1.5 line 709)
chunks = chunk_by_section(text)

# Preview one chunk as a sanity check (Week_05.md line 710)
chunks[2]

## Step 3 · Generate Embeddings for Each Chunk

Pass the list of chunks directly to `generate_embedding` — its S4_02 signature accepts both a single string *and* a list, so we get a **list of embedding vectors** back in a single API call.

**Week_05.md §1.5 reference**: line 718 (`embeddings = generate_embedding(chunks)`).

> [!tip] Batching matters
> Calling `generate_embedding(chunks)` once with a list issues one HTTP request; looping `generate_embedding(c) for c in chunks` issues thirteen. On VoyageAI's free tier the latter triggers rate limits.

In [ ]:
# Step 2: Generate embeddings (list in → list out)  (Week_05.md §1.5 line 718)
embeddings = generate_embedding(chunks)

## Step 4 · Create a Vector Store and Add (Embedding, Chunk) Pairs

Build a fresh `VectorIndex` and insert each embedding together with its source text.

**Week_05.md §1.5 reference**: lines 727–729 — the `for` loop with `store.add_vector(embedding, {"content": chunk})`.

> [!finding] Why store the text, not just the vector?
> The search method returns *documents* (not raw vectors). Keeping `{"content": chunk}` alongside the embedding is what lets the retriever hand back human-readable context for Claude's prompt.

> [!tip] Note on the Skilljar original
> The downloaded notebook called `store.add_documents([...])` (plural), but the `VectorIndex` class in cell 9 only defines `add_document` (singular) + `add_vector`. Week_05.md uses the explicit `for`-loop form below — it matches the class exactly and is clearer for pedagogy.

In [ ]:
# Step 3: Create a vector store and add each (embedding, chunk) pair
# Week_05.md §1.5 lines 727-729 — explicit loop using the add_vector API defined on VectorIndex.
store = VectorIndex()

for embedding, chunk in zip(embeddings, chunks):
    store.add_vector(embedding, {"content": chunk})

## Step 5 · Embed the User Query

Turn the user's natural-language question into a vector in the **same embedding space** as the stored chunks. Only then can cosine distance be meaningful.

**Week_05.md §1.5 reference**: line 740. The canonical demo question is *"What did the software engineering dept do last year?"* — it is not a keyword query; a semantic embedding is precisely what is needed here.

In [ ]:
# Step 4: Embed the user query  (Week_05.md §1.5 line 740)
user_embedding = generate_embedding(
    "What did the software engineering dept do last year?"
)

## Step 6 · Retrieve the Top-k Most Relevant Chunks

`store.search(user_embedding, k=2)` returns the two chunks closest to the query embedding, each paired with its cosine distance.

**Week_05.md §1.5 reference**: lines 748–751.

> [!finding] Expected output
> The top-2 distances should be approximately **0.71** and **0.72**, both pointing to Section 2 *Software Engineering* content (Week_05.md line 811 confirms this). The exact numbers depend on the VoyageAI model version, but the ranking should be stable.

> [!action] Extension — feed top-2 chunks to Claude
> Uncomment and run the optional block below to complete the end-to-end RAG flow (Week_05.md §1.5 "Claude 프롬프트 주입").

In [ ]:
# Step 5: Similarity search — find the 2 most relevant chunks
# Week_05.md §1.5 lines 748-751
results = store.search(user_embedding, 2)

for doc, distance in results:
    print(distance, "\n", doc["content"][0:200], "\n")

# ── Optional: feed the retrieved context to Claude for grounded generation ──
# import anthropic
# anthropic_client = anthropic.Anthropic()
# retrieved = "\n---\n".join(doc["content"] for doc, _ in results)
# response = anthropic_client.messages.create(
#     model="claude-haiku-4-5",
#     max_tokens=1024,
#     system="Answer the user's question using only the provided context. Cite the section you used.",
#     messages=[{
#         "role": "user",
#         "content": f"Context:\n{retrieved}\n\nQuestion: What did the software engineering dept do last year?"
#     }],
# )
# print(response.content[0].text)

## Wrap-up

You have built a working semantic search engine grounded to Claude. The next gap we close in **S4_04** is the *semantic-only blind spot* — queries containing rare literal tokens (e.g., `INC-2023-Q4-011`) that embeddings underweight.

> [!ref] Skilljar L04 (theory) + L05 (code)
> Week_05.md §1.4 "The Full RAG Flow" · §1.5 "Implementing the RAG Flow"